[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/ML_Pipeline.ipynb)


Playbook: [AI / Machine Learning](../../playbooks/ai/ml/README.md)
# ML Pipeline — From Raw Data to Predictions

**The complete Machine Learning pipeline in one notebook.**

Loads call center data → cleans → engineers features → trains models → evaluates → explains with SHAP.

No cloud. No MLflow. Just Python + scikit-learn. This is where every ML project starts.

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

# On Colab: clone the repo to get the data
if not os.path.exists("data") and not os.path.exists("../../data"):
    print("Downloading data from GitHub...")
    !git clone --depth 1 https://github.com/sunilmogadati/systems-in-production.git /tmp/repo 2>/dev/null
    DATA_DIR = "/tmp/repo/data"
elif os.path.exists("../../data/calls.json"):
    DATA_DIR = "../../data"
elif os.path.exists("data/calls.json"):
    DATA_DIR = "data"
else:
    DATA_DIR = os.path.expanduser("~/Downloads/systems-in-production/data")

print(f"Data directory: {DATA_DIR}")

---

## Stage 1: BRONZE — Load Raw Data

Same raw data as the Analytics Pipeline. Both pipelines start from the same source.

In [ ]:
calls_raw = pd.read_json(os.path.join(DATA_DIR, "calls.json"), lines=True)
campaigns_raw = pd.read_csv(os.path.join(DATA_DIR, "campaigns.csv"))
orders_raw = pd.read_csv(os.path.join(DATA_DIR, "orders.csv"))

print(f"calls:     {len(calls_raw):,} records")
print(f"campaigns: {len(campaigns_raw):,} records")
print(f"orders:    {len(orders_raw):,} records")

---

## Stage 2: SILVER — Clean the Data

In [ ]:
calls = calls_raw.copy()

# Flatten nested customer data
if "customer" in calls.columns:
    customer_df = pd.json_normalize(calls["customer"])
    customer_df.columns = ["customer_" + c.replace(".", "_") for c in customer_df.columns]
    calls = pd.concat([calls.drop(columns=["customer"]), customer_df], axis=1)

# Deduplicate
calls = calls.drop_duplicates(subset=["call_id"], keep="first")

# Fix types
calls["date"] = pd.to_datetime(calls["date"], errors="coerce")
calls["start_time"] = pd.to_datetime(calls["start_time"], errors="coerce", utc=True)
calls["end_time"] = pd.to_datetime(calls["end_time"], errors="coerce", utc=True)
calls["dnis"] = calls["dnis"].astype(str)
calls["disposition"] = calls["disposition"].str.strip().str.lower()
calls["channel"] = calls["channel"].str.strip().str.upper()

print(f"Silver calls: {len(calls):,} records")

---

## Stage 3: GOLD (ML) — Feature Engineering

**Key difference from Analytics Gold:** The ML Gold **drops records with missing features.** The analytics Gold keeps them all.

Same Silver layer, different Gold consumers. The DE team keeps every record for accurate counts. The ML team needs clean feature vectors.

In [ ]:
ml_data = calls.copy()

# --- Join campaign info ---
campaigns_raw["dnis"] = campaigns_raw["dnis"].astype(str)
ml_data = ml_data.merge(
    campaigns_raw[["dnis", "campaign_name", "client_name", "channel"]].rename(
        columns={"channel": "campaign_channel"}),
    on="dnis", how="left"
)

# --- Join order info (did the caller place an order?) ---
order_summary = orders_raw.groupby("call_id").agg(
    order_count=("order_id", "count"),
    total_revenue=("total", "sum")
).reset_index()
ml_data = ml_data.merge(order_summary, on="call_id", how="left")
ml_data["order_count"] = ml_data["order_count"].fillna(0).astype(int)
ml_data["total_revenue"] = ml_data["total_revenue"].fillna(0)
ml_data["placed_order"] = (ml_data["order_count"] > 0).astype(int)

print(f"Joined campaign + order data")
print(f"Columns: {list(ml_data.columns)}")

In [ ]:
# --- Traditional features (from existing columns) ---
ml_data["hour_of_day"] = ml_data["start_time"].dt.hour
ml_data["is_weekend"] = (ml_data["start_time"].dt.dayofweek >= 5).astype(int)
ml_data["is_va"] = (ml_data["campaign_channel"] == "VA").astype(int)

# --- Target variable ---
# Churn risk proxy: callback, abandoned, or voicemail = unresolved = churn risk
# In production, actual churn would come from CRM data.
ml_data["is_churn_risk"] = ml_data["disposition"].isin(
    ["callback", "abandoned", "voicemail"]
).astype(int)

print(f"Churn risk rate: {ml_data['is_churn_risk'].mean():.1%}")
print(f"\nDisposition breakdown:")
ml_data["disposition"].value_counts()

In [ ]:
# --- Drop nulls in critical features ---
# Analytics keeps these rows. ML drops them.
critical = ["duration", "hour_of_day"]
before = len(ml_data)
ml_data = ml_data.dropna(subset=critical)
dropped = before - len(ml_data)
print(f"Dropped {dropped} records with missing features")
print(f"ML-ready records: {len(ml_data):,}")

---

## Stage 4: Model Training and Evaluation

Train two models. Compare them honestly. Use the right metrics — not just accuracy.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report
)
from sklearn.preprocessing import StandardScaler

# Select features
feature_cols = ["duration", "hour_of_day", "is_weekend", "is_va",
                "placed_order", "order_count", "total_revenue"]

X = ml_data[feature_cols].fillna(0)
y = ml_data["is_churn_risk"]

print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"Target: is_churn_risk")
print(f"Class balance: {dict(y.value_counts())}")

In [ ]:
# Split (stratify keeps the class ratio the same in train and test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale (Logistic Regression needs this)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")

In [ ]:
# Train and evaluate both models
models = {
    "Logistic Regression": LogisticRegression(random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
}

results = []

for name, model in models.items():
    if "Logistic" in name:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring="f1")
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="f1")

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_proba) if len(y_test.unique()) > 1 else 0

    results.append({"Model": name, "Accuracy": acc, "Precision": prec,
                     "Recall": rec, "F1": f1, "ROC-AUC": auc})

    print(f"\n{name}:")
    print(f"  Accuracy:  {acc:.3f}")
    print(f"  Precision: {prec:.3f}  (of flagged churn risks, how many are real?)")
    print(f"  Recall:    {rec:.3f}  (of actual risks, how many caught?)")
    print(f"  F1:        {f1:.3f}")
    print(f"  ROC-AUC:   {auc:.3f}")
    print(f"  5-Fold CV: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

In [ ]:
# Model comparison table
results_df = pd.DataFrame(results).set_index("Model")
print("MODEL COMPARISON")
results_df.round(3)

### The Honest Assessment

Both models perform poorly. This is **not a bug — it is the lesson.**

The features available (duration, hour, weekend, channel, order info) do not predict churn risk. Knowing a call happened at 3 PM on a weekday via the VA channel tells the model almost nothing about whether the customer will churn.

**What WOULD predict churn?**
- The call transcript ("I'm frustrated, this is the third time I've called")
- Customer sentiment (angry → churn risk)
- Call history (5 calls in 2 weeks = unresolved problem)
- The reason for the call (billing dispute vs product question)

These are **AI-derived features** — they require an LLM (Large Language Model) to analyze the transcript and extract the signal. No spreadsheet column captures "frustrated customer who mentioned a competitor."

**The principle: bad features produce bad models, regardless of the algorithm.** No amount of hyperparameter tuning fixes missing signal. The next step is not a better model — it is better features.

---

## Stage 5: Explainability — SHAP

Even a weak model can be explained. SHAP shows which features the model relied on — and confirms that none of them have strong signal.

In [ ]:
try:
    import shap

    rf_model = models["Random Forest"]
    explainer = shap.TreeExplainer(rf_model)
    shap_values = explainer.shap_values(X_test)

    # Handle different SHAP output formats
    if isinstance(shap_values, list):
        shap_vals = shap_values[1]
    elif shap_values.ndim == 3:
        shap_vals = shap_values[:, :, 1]
    else:
        shap_vals = shap_values

    # Feature importance
    importance = pd.DataFrame({
        "Feature": feature_cols,
        "Mean |SHAP|": np.abs(shap_vals).mean(axis=0)
    }).sort_values("Mean |SHAP|", ascending=False)

    print("Feature Importance (SHAP — what the model relies on):")
    print("=" * 50)
    max_val = importance["Mean |SHAP|"].max()
    for _, row in importance.iterrows():
        bar_len = int(30 * row["Mean |SHAP|"] / max_val) if max_val > 0 else 0
        bar = "█" * bar_len
        print(f"  {row['Feature']:20s} {row['Mean |SHAP|']:.4f} {bar}")

except ImportError:
    print("SHAP not installed. Run: pip install shap")

In [ ]:
# Explain one specific prediction
try:
    idx = 0
    pred = rf_model.predict(X_test.iloc[[idx]])[0]
    proba = rf_model.predict_proba(X_test.iloc[[idx]])[0][1]
    actual = y_test.iloc[idx]

    print(f"Example prediction (test row {idx}):")
    print(f"  Predicted: {'CHURN RISK' if pred == 1 else 'NO RISK'} ({proba:.1%} probability)")
    print(f"  Actual:    {'CHURN RISK' if actual == 1 else 'NO RISK'}")
    print(f"  Why:")
    for feat, val, sv in zip(feature_cols, X_test.iloc[idx], shap_vals[idx]):
        direction = "↑ risk" if sv > 0 else "↓ risk"
        print(f"    {feat:20s} = {val:8.1f}  →  SHAP: {sv:+.4f} ({direction})")
except:
    print("SHAP not available for individual explanation.")

---

## Summary and Next Steps

In [ ]:
print("ML PIPELINE COMPLETE")
print("=" * 50)
print(f"Bronze: {len(calls_raw):,} raw records")
print(f"Silver: {len(calls):,} cleaned records")
print(f"Gold (ML): {len(ml_data):,} feature-ready records")
print(f"Best model: {results_df['F1'].idxmax()} (F1: {results_df['F1'].max():.3f})")
print()
print("KEY LESSON: The models are weak because the features are weak.")
print("Duration and hour_of_day don't predict churn. The transcript does.")
print()
print("NEXT STEPS:")
print("  1. AI-derived features: classify transcripts with an LLM")
print("     (disposition reason, sentiment, customer intent)")
print("  2. Better target: actual churn from CRM, not simulated from dispositions")
print("  3. More data: 500 calls is small. 50,000 would give the model more to learn.")
print("  4. Production: MLflow for tracking, FastAPI for serving, monitoring for drift")